In [3]:
import pandas as pd
import numpy as np
 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

data = pd.read_csv("pos_tags.csv")
print(data.head())



   Unnamed: 0    word pos_tag
0           0      aa      NN
1           1     aaa      NN
2           2     aah      NN
3           3   aahed     VBN
4           4  aahing     VBG


In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
data=pd.read_csv("pos_tags.csv",nrows=1000)
print(data.head())
sentences=[]
temp=[]
for _,row in data.iterrows():
    word=row["word"]
    tag=row["tag"]
    temp.append((word,tag))
    if(len(temp))==20:
        sentences.append(temp)
        temp=[]
print(sentences)

train_data,test_data=train_test_split(sentences,test_size=0.2,random_state=42)
#create vocabulary
vocabulary = set()

tags=set()
for sentence in train_data:
    for word,tag in sentence:
        vocabulary.add(word.lower())
        tags.add(tag)
word_to_index={
word:i
for i,word in enumerate(vocabulary)
}

tag_to_index={
    tag:i
    for i,tag in enumerate(tags)
}

index_to_tag={
    i:tag
    for tag,i in tag_to_index.items()
}
V=len(vocabulary)
T=len(tags)
print("\nVocabulary Size:", V)
print("Number of POS tags:",T)

#HMM Matrices
#Inital , transition,emission probability
#initial - probability(starting tag)
#transition - Relationship between tow pos tags - P(current tag|previos tag)
#emission - Relation between word and tag –p(word|tag)

initial=np.ones(T)

transition=np.ones((T,T))

emission=np.ones((T,V))

#count how many times each tag appears
for sentence in train_data:
# first tag
    first_tag=sentence[0][1]
    initial[
        tag_to_index[first_tag] ] += 1
    for i,(word,tag) in enumerate(sentence):
        word=word.lower()
        tag_index=tag_to_index[tag]

#emission count
if word in word_to_index:
    emission[tag_index,word_to_index[word] ] += 1
#Transition Count
if i>0:
    previous_tag=sentence[i-1][1]
    transition[ tag_to_index[previous_tag], tag_index] += 1

#convert the account to probability value
initial /= initial.sum()
transition /= transition.sum( axis=1, keepdims=True )
emission /= emission.sum( axis=1, keepdims=True )


#convert to log
log_initial=np.log(initial)
log_transition=np.log(transition)
log_emission=np.log(emission)
#viterbi Algorithm
def viterbi(sentence):
    n=len(sentence)
    dp=np.zeros((T,n)) 
    backpointer=np.zeros( (T,n),dtype=int)
    # First word
    word=sentence[0].lower()
    if word in word_to_index:
        emit=log_emission[:,word_to_index[word] ]
    else:
        emit=np.log( np.ones(T)*1e-10)
    dp[:,0]=(log_initial+emit)

    #remaining words
    for i in range(1,n):
        word=sentence[i].lower()
        if word in word_to_index:
            emit=log_emission[:, word_to_index[word] ]
        else:
            emit=np.log(np.ones(T)*1e-10)
    #vector calcualtion
    scores=( dp[:,i-1][:,None] + log_transition )
    backpointer[:,i]=np.argmax(scores,axis=0)
    dp[:,i]=(np.max(scores,   axis=0) +emit)
    #backtracking
    best=np.argmax( dp[:,-1])
    result=[best]
    for i in range(n-1,0,-1):
        best=backpointer[
            best,i
        ]
        result.append(best)
    result.reverse()
    return [
        index_to_tag[i]
        for i in result
    ]

    sentence="Artificial intelligence improves healthcare systems".split()
    prediction=viterbi(sentence)
    print("\nPrediction")
    print("----------------")
    for word,tag in zip(sentence,prediction):
        print(
        word,
        "---->",
        tag)

   sentence_id    word  tag
0            0      aa   NN
1            1     aaa   NN
2            2     aah   NN
3            3   aahed  VBN
4            4  aahing  VBG
[[('aa', 'NN'), ('aaa', 'NN'), ('aah', 'NN'), ('aahed', 'VBN'), ('aahing', 'VBG'), ('aahs', 'NN'), ('aal', 'NN'), ('aalii', 'NN'), ('aaliis', 'NN'), ('aals', 'NNS'), ('aam', 'NN'), ('aani', 'NN'), ('aardvark', 'NN'), ('aardvarks', 'NNS'), ('aardwolf', 'NN'), ('aardwolves', 'NNS'), ('aargh', 'NN'), ('aaron', 'NN'), ('aaronic', 'NN'), ('aaronical', 'JJ')], [('aaronite', 'NN'), ('aaronitic', 'JJ'), ('aarrgh', 'NN'), ('aarrghh', 'NN'), ('aaru', 'NN'), ('aas', 'NN'), ('aasvogel', 'NN'), ('aasvogels', 'NNS'), ('ab', 'NN'), ('aba', 'NN'), ('ababdeh', 'NN'), ('ababua', 'NN'), ('abac', 'NN'), ('abaca', 'NN'), ('abacay', 'NN'), ('abacas', 'NN'), ('abacate', 'NN'), ('abacaxi', 'NN'), ('abaci', 'NN'), ('abacinate', 'NN')], [('abacination', 'NN'), ('abacisci', 'NN'), ('abaciscus', 'NN'), ('abacist', 'NN'), ('aback', 'NN'), ('abacli',

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# -----------------------------
# Load Dataset
# -----------------------------
data = pd.read_csv("pos_tags.csv")

print("Dataset Preview:")
print(data.head())

# -----------------------------
# Create Sentences
# -----------------------------
sentences = []
temp = []

for _, row in data.iterrows():
    word = row["word"]
    tag = row["tag"]

    temp.append((word, tag))

    # Assuming each sentence contains 20 words
    if len(temp) == 20:
        sentences.append(temp)
        temp = []

print("\nTotal Sentences:", len(sentences))

# -----------------------------
# Train-Test Split
# -----------------------------
train_data, test_data = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

# -----------------------------
# Create Vocabulary and Tags
# -----------------------------
vocabulary = set()
tags = set()

for sentence in train_data:
    for word, tag in sentence:
        vocabulary.add(word.lower())
        tags.add(tag)

word_to_index = {
    word: i
    for i, word in enumerate(vocabulary)
}

tag_to_index = {
    tag: i
    for i, tag in enumerate(tags)
}

index_to_tag = {
    i: tag
    for tag, i in tag_to_index.items()
}

V = len(vocabulary)
T = len(tags)

print("\nVocabulary Size:", V)
print("Number of POS Tags:", T)

# -----------------------------
# HMM Matrices
# -----------------------------
initial = np.ones(T)

transition = np.ones((T, T))

emission = np.ones((T, V))

# -----------------------------
# Count Initial, Transition,
# and Emission Probabilities
# -----------------------------
for sentence in train_data:

    # Initial tag
    first_tag = sentence[0][1]
    initial[tag_to_index[first_tag]] += 1

    for i, (word, tag) in enumerate(sentence):

        word = word.lower()
        tag_index = tag_to_index[tag]

        # Emission Count
        if word in word_to_index:
            emission[tag_index, word_to_index[word]] += 1

        # Transition Count
        if i > 0:
            previous_tag = sentence[i - 1][1]

            transition[
                tag_to_index[previous_tag],
                tag_index
            ] += 1

# -----------------------------
# Convert Counts to Probabilities
# -----------------------------
initial /= initial.sum()

transition /= transition.sum(axis=1, keepdims=True)

emission /= emission.sum(axis=1, keepdims=True)

# -----------------------------
# Convert to Log Probabilities
# -----------------------------
log_initial = np.log(initial)

log_transition = np.log(transition)

log_emission = np.log(emission)

# -----------------------------
# Viterbi Algorithm
# -----------------------------
def viterbi(sentence):

    n = len(sentence)

    dp = np.zeros((T, n))

    backpointer = np.zeros((T, n), dtype=int)

    # First Word
    word = sentence[0].lower()

    if word in word_to_index:
        emit = log_emission[:, word_to_index[word]]
    else:
        emit = np.full(T, np.log(1e-10))

    dp[:, 0] = log_initial + emit

    # Remaining Words
    for i in range(1, n):

        word = sentence[i].lower()

        if word in word_to_index:
            emit = log_emission[:, word_to_index[word]]
        else:
            emit = np.full(T, np.log(1e-10))

        scores = dp[:, i - 1][:, None] + log_transition

        backpointer[:, i] = np.argmax(scores, axis=0)

        dp[:, i] = np.max(scores, axis=0) + emit

    # Backtracking
    best = np.argmax(dp[:, -1])

    result = [best]

    for i in range(n - 1, 0, -1):
        best = backpointer[best, i]
        result.append(best)

    result.reverse()

    predicted_tags = [
        index_to_tag[i]
        for i in result
    ]

    return predicted_tags

# -----------------------------
# Test Sentence
# -----------------------------
sentence = "Artificial intelligence improves healthcare systems".split()

prediction = viterbi(sentence)

print("\nPrediction")
print("-" * 30)

for word, tag in zip(sentence, prediction):
    print(f"{word:15} --> {tag}")

Dataset Preview:
   sentence_id    word  tag
0            0      aa   NN
1            1     aaa   NN
2            2     aah   NN
3            3   aahed  VBN
4            4  aahing  VBG

Total Sentences: 18505

Vocabulary Size: 296080
Number of POS Tags: 25

Prediction
------------------------------
Artificial      --> NN
intelligence    --> NN
improves        --> NN
healthcare      --> NN
systems         --> NN
